In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

dataset_files = list(Path(".").rglob("Mall_Customers.csv"))

if not dataset_files:
    raise FileNotFoundError("Extract the practical datasets ZIP beside this notebook.")

df = pd.read_csv(dataset_files[0])

X = df[["Annual Income (k$)", "Spending Score (1-100)"]]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Dataset shape:", df.shape)
display(df.head())

In [ ]:
cluster_range = range(2, 8)
inertia_values = []
silhouette_values = []

for n_clusters in cluster_range:
    kmeans = KMeans(
        n_clusters=n_clusters,
        random_state=42,
        n_init=10
    )

    cluster_labels = kmeans.fit_predict(X_scaled)

    inertia_values.append(kmeans.inertia_)
    silhouette_values.append(
        silhouette_score(X_scaled, cluster_labels)
    )

results = pd.DataFrame({
    "Number of Clusters": list(cluster_range),
    "Inertia": inertia_values,
    "Silhouette Score": silhouette_values
})

display(results.round(4))

plt.figure(figsize=(7, 4))
plt.plot(list(cluster_range), inertia_values, marker="o")
plt.xlabel("Number of Clusters")
plt.ylabel("Within-Cluster Sum of Squares")
plt.title("Elbow Method")
plt.grid(alpha=0.25)
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(list(cluster_range), silhouette_values, marker="o")
plt.xlabel("Number of Clusters")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Analysis")
plt.grid(alpha=0.25)
plt.show()

In [ ]:
optimal_k = 5

kmeans = KMeans(
    n_clusters=optimal_k,
    random_state=42,
    n_init=10
)

df["Cluster"] = kmeans.fit_predict(X_scaled)

cluster_score = silhouette_score(X_scaled, df["Cluster"])

print("Optimal number of clusters:", optimal_k)
print("Silhouette Score:", round(cluster_score, 4))
print("Cluster Distribution:")
print(df["Cluster"].value_counts().sort_index())

plt.figure(figsize=(8, 5))

for cluster in sorted(df["Cluster"].unique()):
    cluster_data = df[df["Cluster"] == cluster]

    plt.scatter(
        cluster_data["Annual Income (k$)"],
        cluster_data["Spending Score (1-100)"],
        label=f"Cluster {cluster}",
        s=40
    )

centers = scaler.inverse_transform(kmeans.cluster_centers_)

plt.scatter(
    centers[:, 0],
    centers[:, 1],
    color="black",
    marker="X",
    s=180,
    label="Centroids"
)

plt.xlabel("Annual Income (k$)")
plt.ylabel("Spending Score (1-100)")
plt.title("K-Means Customer Segmentation")
plt.legend()
plt.grid(alpha=0.25)
plt.show()